In [1]:


import jax 
import jax.numpy as jnp


import haiku as hk
import optax

from probjax.nn.transformers import Transformer
from probjax.nn.tokenizer import scalarize, ScalarTokenizer
from probjax.nn.helpers import GaussianFourierEmbedding
from probjax.nn.loss_fn import denoising_score_matching_loss

from probjax.distributions.sde import VPSDE, BaseSDE
from probjax.distributions import Normal, Independent
from probjax.distributions.transformed_distribution import TransformedDistribution
from probjax.distributions.discrete import Empirical

from probjax.utils.sdeint import sdeint

from functools import partial




I0000 00:00:1701101684.979343  301120 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.
2023-11-27 17:14:47.308739: W external/xla/xla/service/gpu/nvptx_compiler.cc:708] The NVIDIA driver's CUDA version is 11.7 which is older than the ptxas CUDA version (11.8.89). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [ ]:
from sbibm import get_task

task = get_task("two_moons")
prior = task.get_prior_dist()
simulator = task.get_simulator()

thetas = prior.sample((1000,))
xs = simulator(thetas)

thetas = jnp.array(thetas)
xs = jnp.array(xs)


In [516]:
with jax.default_device(jax.devices()[1]):
    print(jax.local_devices())

[gpu(id=0), gpu(id=1), gpu(id=2), gpu(id=3), gpu(id=4), gpu(id=5), gpu(id=6), gpu(id=7)]


In [504]:


def conditional_mlp(output_dim:int, hidden_dim:int = 100, num_hidden:int=8, activation=jax.nn.gelu, layer_norm:bool = True, output_scale_fn=None):
    
    if output_scale_fn is None:
        output_scale_fn = lambda t, x: x
    
    def score_net(t, x, context):
        
        #x, context = jnp.broadcast_arrays(x, context)
        x = jnp.concatenate([x, context], axis=-1)
        
        time_embedding = GaussianFourierEmbedding(hidden_dim)(t[...,None])
        h = activation(hk.Linear(hidden_dim)(x) + time_embedding)
        
        
        for _ in range(num_hidden - 1):
            h_new = hk.Linear(hidden_dim)(h)
            h_new += time_embedding
            h = activation(h_new)
            
            if layer_norm:
                h = hk.LayerNorm(axis=-1, create_scale=True, create_offset=True)(h)
            
        out = hk.Linear(output_dim)(h)
        out = output_scale_fn(t, out)
        return out
    
    init_fn, apply_fn = hk.without_apply_rng(hk.transform(score_net))
    return init_fn, apply_fn

def init_sde_related(data, name="vpsde", **kwargs):
    # VPSDE 
    if name.lower() == "vpsde":
        p0 = Independent(Empirical(data), 1)
        beta_max = kwargs.get("beta_max",10.)
        beta_min = kwargs.get("beta_min", 0.01)
        sde = VPSDE(p0, beta_max=beta_max, beta_min=beta_min)
        T_max = kwargs.get("T_max", 1.)
        T_min = kwargs.get("T_min", 1e-5)

        # Train weight function
        def weight_fn(t):
            t = t.reshape(-1, 1)
            return jnp.clip(1-jnp.exp(-0.5 * (beta_max - beta_min) * t**2 - beta_min * t) ,a_min = 1e-4)
        
        # Model output scale function
        def output_scale_fn(t, x):
            scale = jnp.sqrt(jnp.sum(sde.marginal_variance(t[..., None], x0=jnp.ones_like(x)), -1))
            return jnp.clip(1/scale[..., None],a_min=0.01, a_max=100.) * x
        
    else:
        raise NotImplementedError()
    
    return sde, T_min, T_max, weight_fn, output_scale_fn



def run_train_conditional_score_model(key, params, opt_state, data, num_epochs, num_steps, batch_size,  update, print_every=100):
    num_devices = jax.device_count()
    batch_size_per_device = batch_size // num_devices
    # Replicated for multiple devices
    replicated_params = jax.tree_map(lambda x: jnp.array([x] * num_devices), params)
    replicated_opt_state = jax.tree_map(lambda x: jnp.array([x] * num_devices), opt_state)

    for j in range(num_epochs):
        l = 0
        for i in range(num_steps):
            key, key_batch, key_update = jax.random.split(key, 3)
            data_batch = jax.random.choice(key_batch,data, shape=(num_devices, batch_size_per_device, ), axis=0, replace=True)
            loss, replicated_params, replicated_opt_state = update(replicated_params, replicated_opt_state, jax.random.split(key_update, (num_devices,)), data_batch)
            l += loss[0] /num_steps
        if (j % print_every) == 0:     
            print("Train loss: ",l)
            
    params = jax.tree_map(lambda x: x[0], replicated_params)
    
    return params


class NPSE:
    def __init__(self, params, model_fn, sde, model_init_params={}, sde_init_params={}) -> None:
        self.params = params
        self.model_fn = model_fn
        self.sde = sde
        
        # For sampling
        self.T_min = sde_init_params["T_min"]
        self.T_max = sde_init_params["T_max"]
        self.marginal_end_std = sde.marginal_stddev(jnp.array([self.T_min]))
        self.marginal_end_mean = sde.marginal_mean(jnp.array([self.T_max]))
        
        # For pickle 
        self.model_init_params = model_init_params
        self.sde_init_params = sde_init_params
        
        
        
    def sample(self, key,num_samples, x_o, num_steps=500, **kwargs):
        key1, key2 = jax.random.split(key, 2)
        drift, diffusion = self._init_backward_sde(x_o)
        x_T = jax.random.normal(key1, (num_samples,) + self.sde.event_shape) * self.marginal_end_std + self.marginal_end_mean
        keys = jax.random.split(key2, (num_samples,))
        ys = jax.vmap(lambda *args: sdeint(*args, noise_type="diagonal",**kwargs), in_axes= (0, None, None, 0, None), out_axes=0)(keys, drift, diffusion, x_T, jnp.linspace(0., self.T_max-self.T_min, num_steps))
        return ys[:, -1, ...]
    
    def log_prob(self, val, x_o, **kwargs):
        # Add backward ode to compute log_prob
        raise NotImplementedError()
    
        
    def _init_backward_sde(self, x_o):
        def drift_backward(t, x):
            t = (self.T_max-t)
            score = self.model_fn(self.params, jnp.atleast_1d(t), x, jnp.squeeze(x_o))
            drift = self.sde.drift(t, x)  - self.sde.diffusion(t, x)**2 * score
            return -drift.reshape(x.shape)
        
        def diffusion_backward(t, x):
            t = (self.T_max-t)
            return self.sde.diffusion(t, x).reshape(x.shape)
        
        return drift_backward, diffusion_backward
    
    def __getstate__(self) -> object:
        state = self.__dict__.copy()
        state["model_fn"] = None 
        state["sde"] = None
        return state    
    
    def __setstate__(self, state):
        self.__dict__.update(state)
        self.sde, self.T_min, self.T_max, _, output_scale_fn = init_sde_related(**self.sde_init_params)
        _, self.model_fn = conditional_mlp(output_scale_fn=output_scale_fn,**self.model_init_params)
    
    
                 


In [505]:
data = jnp.hstack([thetas, xs])
theta_dim = thetas.shape[-1]
x_dim = xs.shape[-1]

rng = jax.random.PRNGKey(0)

sde, T_min,T_max, weight_fn, output_scale_fn = init_sde_related(thetas)

In [457]:
init_fn, model_fn = conditional_mlp(theta_dim, output_scale_fn=output_scale_fn)
rng, rng_init = jax.random.split(rng)
params = init_fn(rng_init, jnp.ones((10,)), thetas[:10], xs[:10])

In [478]:
total_number_steps = int(data.shape[0])
batch_size = 5000
num_steps = data.shape[0] // batch_size + 1

num_epochs = total_number_steps // num_steps 
print_every = num_epochs // 10
learning_rate = 5e-4
schedule = optax.linear_schedule(learning_rate, 0., total_number_steps//2, total_number_steps//2)
optimizer = optax.chain(optax.adaptive_grad_clip(5.), optax.adam(schedule))
opt_state = optimizer.init(params)

In [479]:
@jax.jit
def loss_fn(params, key, data):
    thetas, xs = jnp.split(data, [theta_dim,], axis=-1)
    key_times, key_loss = jax.random.split(key,2)
    times = jax.random.uniform(key_times, (data.shape[0],), minval=T_min, maxval =T_max)
    loss = denoising_score_matching_loss(params, key_loss, times, thetas, None, xs, model_fn = model_fn, mean_fn = sde.marginal_mean, std_fn=sde.marginal_stddev, weight_fn=weight_fn, axis=-1)
    return loss

@partial(jax.pmap, axis_name="num_devices")
def update(params, opt_state, key, data):
    loss, grads = jax.value_and_grad(loss_fn)(params, key, data)

    loss = jax.lax.pmean(loss, axis_name="num_devices")
    grads = jax.lax.pmean(grads, axis_name="num_devices")
    
    updates, opt_state = optimizer.update(grads, opt_state, params=params)
    params = optax.apply_updates(params, updates)
    return loss, params, opt_state


In [480]:
rng, rng_train = jax.random.split(rng)
params = run_train_conditional_score_model(rng_train, params, opt_state, data, num_epochs, num_steps, batch_size, update, print_every=print_every)

Train loss:  1.7798154


KeyboardInterrupt: 

In [510]:
model = NPSE(params, model_fn, sde, sde_init_params={"data": thetas, "T_max": T_max, "T_min": T_min}, model_init_params={"output_dim": theta_dim})

In [507]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
model = pickle.load(open("model.pkl", "rb"))

In [508]:
from sbi.analysis.sbc import c2st
import torch
import numpy as np

In [511]:

c2sts = []
for i in range(1,11):
    x_o = task.get_observation(i)
    samples_post = task.get_reference_posterior_samples(i)
    x_o = jnp.array(x_o)
    samples_est = model.sample(jax.random.PRNGKey(i), 10000, x_o)
    metric = c2st(torch.tensor(np.array(samples_est)), samples_post)
    print("C2ST: ", metric)

C2ST:  tensor([0.5997])
C2ST:  tensor([0.5882])
C2ST:  tensor([0.5702])
C2ST:  tensor([0.5838])
C2ST:  tensor([0.5619])
C2ST:  tensor([0.5617])
C2ST:  tensor([0.5708])
C2ST:  tensor([0.5737])
C2ST:  tensor([0.5892])
C2ST:  tensor([0.6079])


In [ ]:
def train_conditional_score_model(task, thetas,xs, method_cfg, rng):

    device = method_cfg.device
    sde_params = method_cfg.sde_params
    model_params = method_cfg.model_params
    train_params = method_cfg.params_train
    
    
    # Data
    data = jnp.hstack([thetas, xs])
    theta_dim = thetas.shape[-1]
    x_dim = xs.shape[-1]
    
    # Initialize stuff
    sde, T_min,T_max, weight_fn, output_scale_fn = init_sde_related(data, **sde_params)
    init_fn, model_fn = conditional_mlp(theta_dim, output_scale_fn=output_scale_fn, **model_params)
    
    rng, rng_init = jax.random.split(rng)
    params = init_fn(rng_init, jnp.ones((10,)), thetas[:10], xs[:10])

    
    num_epochs = train_params.max_num_epochs
    batch_size = train_params.training_batch_size
    num_devices = jax.device_count()
    batch_size_per_device = batch_size // num_devices
    num_steps = data.shape[0] // batch_size + 1
    
    total_number_steps = num_epochs * num_steps
    learning_rate = train_params.learning_rate
    schedule = optax.linear_schedule(learning_rate, 0., total_number_steps//2, total_number_steps//2)
    optimizer = optax.chain(optax.adaptive_grad_clip(train_params.clip_max_norm), optax.adam(schedule))
    opt_state = optimizer.init(params)
    
    @jax.jit
    def loss_fn(params, key, data):
        thetas, xs = jnp.split(data, 2, axis=-1)
        key_times, key_loss = jax.random.split(key,2)
        times = jax.random.uniform(key_times, (data.shape[0],), minval=T_min, maxval =T_max)
        loss = denoising_score_matching_loss(params, key_loss, times, thetas, None, xs, model_fn = model_fn, mean_fn = sde.marginal_mean, std_fn=sde.marginal_stddev, weight_fn=weight_fn, axis=-1)
        return loss

    @partial(jax.pmap, axis_name="num_devices")
    def update(params, opt_state, key, data):
        loss, grads = jax.value_and_grad(loss_fn)(params, key, data)

        loss = jax.lax.pmean(loss, axis_name="num_devices")
        grads = jax.lax.pmean(grads, axis_name="num_devices")
        
        updates, opt_state = optimizer.update(grads, opt_state, params=params)
        params = optax.apply_updates(params, updates)
        return loss, params, opt_state
    
    
    
    params = run_train_conditional_score_model(rng, params, opt_state, data, num_epochs, num_steps, batch_size_per_device, num_devices, update, print_every=100)
    
    
    marginal_end_std = sde.marginal_stddev(jnp.array([T_max]))[..., : theta_dim]
    marginal_end_mean = sde.marginal_mean(jnp.array([T_max]))[..., : theta_dim]
    
    def init_backward_sde(x_o):
    
        def score_fn(t, theta):
            theta, observed = jnp.broadcast_arrays(theta, x_o)
            score = model_fn(params, jnp.atleast_1d(t), theta, observed)
            return score.reshape(theta.shape)

        def drift_backwards(t, theta):
            t = (1- t)
            score = score_fn(t, theta)
            drift = sde.drift(t, theta) - sde.diffusion(t, theta)**2 * score 
            drift = -drift
            return drift.reshape(theta.shape)

        def diffusion_backwards(t, theta):
            t = (1-t)
            diffusion = sde.diffusion(t, theta) 
            return diffusion

        return drift_backwards, diffusion_backwards
    
    def sample_fn(key, shape, x_o):
        key1, key2 = jax.random.split(key, 2)
        drift, diffusion = init_backward_sde(x_o)
        x_T = jax.random.normal(key1, shape + (theta_dim,)) * marginal_end_std + marginal_end_mean
        keys = jax.random.split(key2, shape)
        ys = jax.vmap(lambda *args: sdeint(*args, noise_type="diagonal"), in_axes= (0, None, None, 0, None), out_axes=0)(keys, drift, diffusion, x_T, jnp.linspace(0., 1-T_min, 500))
        return ys[:, -1, ...]
    
    
    return params, model_fn, sample_fn, sde, T_min, T_max